# FashionGraph — Phase 5, Track B: fine-tune the image encoder on Colab

The M4 can only *reweight* frozen FashionSigLIP features (linear projection) — and that
flatlined at / below the base (top-1 0.411). To actually improve cross-collection designer
grounding we must **change the features**: partially unfreeze the SigLIP image tower and
fine-tune it with the **KG-concept-weighted supervised-contrastive loss**. That needs a GPU
— this notebook.

It reuses the *already-tested* repo functions (`load_supervision`, `build_designer_weights`,
`split_groups_three`, `designer_topk`, `bootstrap_ci`); the only new code is the encoder
fine-tune loop, kept in the notebook so you can tweak it.

**Before you run**, put on your Google Drive (folder name is configurable below):
```
MyDrive/FashionGraphData/vogue_runway/...      (the runway images + .json sidecars)
MyDrive/FashionGraphData/fashion_kg.sqlite     (the built knowledge graph)
```
Then: **Runtime → Change runtime type → GPU (T4)**, and Run all.

The runway imagery is research/thesis-use only — do not redistribute.

## 1. GPU check

In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime > Change runtime type > GPU'

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Config — set these to match your setup

In [ ]:
# --- where your data lives on Drive ---
DRIVE_DATA_DIR = '/content/drive/MyDrive/FashionGraphData'   # holds vogue_runway/ + fashion_kg.sqlite
OUTPUT_DIR     = '/content/drive/MyDrive/FashionGraphData/phase5_out'  # weights + results saved here

# --- how to get the repo code ---
# Option A: clone from GitHub (set your repo URL; use a token if private).
REPO_URL  = 'https://github.com/<YOUR_USER>/fashiongraph.git'   # <-- EDIT ME
REPO_DIR  = '/content/fashiongraph'
# Option B: if the repo isn't on GitHub, zip it, put FashionGraphData/fashiongraph.zip on Drive,
#          and set USE_DRIVE_ZIP = True (it will unzip instead of cloning).
USE_DRIVE_ZIP = False
REPO_ZIP  = f'{DRIVE_DATA_DIR}/fashiongraph.zip'

# --- model + training hyper-parameters ---
MODEL_NAME      = 'Marqo/marqo-fashionSigLIP'
SUPERVISION     = 'both'      # 'concepts' | 'labels' | 'both'  (both = full ablation)
UNFREEZE_BLOCKS = 2           # how many of the last transformer blocks to unfreeze
EPOCHS          = 15
BATCH           = 64
LR              = 2e-5        # low — we are nudging a pretrained tower
TEMP            = 0.1
WEIGHT_DECAY    = 1e-4
PATIENCE        = 4           # early-stop on val designer top-1
TEST_FRAC       = 0.2
VAL_FRAC        = 0.2         # a bit larger than local, to steady model selection
SEED            = 42
IMG_SIZE        = 256

## 4. Get the repo code + install deps

In [ ]:
import os, sys, subprocess

if USE_DRIVE_ZIP:
    os.makedirs(REPO_DIR, exist_ok=True)
    subprocess.run(['unzip', '-oq', REPO_ZIP, '-d', '/content'], check=True)
    # if the zip contains a top folder, adjust REPO_DIR accordingly
else:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=False)

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print('repo at', REPO_DIR)

In [ ]:
# Colab already has torch (+CUDA). Add the fashion embedder + settings deps.
!pip -q install open_clip_torch pydantic-settings 2>/dev/null
print('deps ok')

## 5. Load the model + the runway images (once, into RAM)

In [ ]:
import json, copy, logging
from pathlib import Path
import numpy as np
import torch
import open_clip
from PIL import Image

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

hub = f'hf-hub:{MODEL_NAME}'
model, _, preprocess = open_clip.create_model_and_transforms(hub)
model = model.to(device)
INIT_STATE = copy.deepcopy(model.state_dict())   # to reset between ablation arms
print('loaded', MODEL_NAME)

In [ ]:
# Walk the runway sidecars (same order build_runway_index uses), load + resize images.
root = Path(DRIVE_DATA_DIR) / 'vogue_runway'
jsons = sorted(root.rglob('*.json'))
assert jsons, f'No runway JSON under {root} — check DRIVE_DATA_DIR.'

pil_images, metas = [], []
for jp in jsons:
    try:
        info = json.loads(jp.read_text(encoding='utf-8'))
    except Exception:
        continue
    img_path = jp.with_suffix('.png')
    if not img_path.exists():
        rec = info.get('image_path')
        cand = root.parent.parent / rec if rec else None   # fallback
        if cand and cand.exists():
            img_path = cand
        else:
            continue
    try:
        im = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    except Exception:
        continue
    pil_images.append(im)
    metas.append({
        'source': 'runway',
        'designer': info.get('designer', ''),
        'show': info.get('show', ''),
        'season': info.get('season', ''),
        'type': info.get('type', ''),
        'title': f"{info.get('designer','')} \u2014 {info.get('show','')}",
    })
print(f'loaded {len(pil_images)} runway images')

## 6. Supervision, splits, and the embed helpers (reuses tested repo code)

In [ ]:
from fg.vision.index import VisualIndex
from fg.kg.store import KnowledgeGraph
from fg.training.alignment_pairs import load_supervision
from fg.training.train_alignment import (
    build_designer_weights, split_groups_three, designer_topk, bootstrap_ci,
)

KG_PATH = str(Path(DRIVE_DATA_DIR) / 'fashion_kg.sqlite')
kg = KnowledgeGraph(KG_PATH)

@torch.no_grad()
def embed_all(net, bs=64):
    net.eval()
    outs = []
    for i in range(0, len(pil_images), bs):
        x = torch.stack([preprocess(im) for im in pil_images[i:i+bs]]).to(device)
        f = net.encode_image(x)
        f = f / f.norm(dim=-1, keepdim=True).clamp(min=1e-8)
        outs.append(f.float().cpu().numpy())
    return np.concatenate(outs, 0)

# Base embeddings (fresh model) → build an index so load_supervision can read metadata.
base_emb = embed_all(model)
index = VisualIndex(base_emb, metas)
records = load_supervision(index, kg)
designers = np.array([r.designer for r in records])
fit_rows, val_rows, test_rows = split_groups_three(records, TEST_FRAC, VAL_FRAC, SEED)
ref_rows = np.concatenate([fit_rows, val_rows])
print(f'split: fit={len(fit_rows)} val={len(val_rows)} test={len(test_rows)} (by collection)')

base = designer_topk(base_emb, designers, ref_rows, test_rows)
print('BASE test top1=%.3f top3=%.3f top5=%.3f' % (base['top1'], base['top3'], base['top5']))

## 7. The encoder fine-tune loop (the new part)

Mini-batch supervised-contrastive on `encode_image` outputs. Positive-pair weights come from
the KG (`labels` = same-designer only; `concepts` = same-designer **plus** concept-overlapping
designers by Jaccard). We unfreeze the last `UNFREEZE_BLOCKS` transformer blocks (+ the final
norm/proj) and train with a low LR, early-stopping on **val** designer top-1.

In [ ]:
def visual_blocks(net):
    v = net.visual
    if hasattr(v, 'trunk') and hasattr(v.trunk, 'blocks'):
        return list(v.trunk.blocks)            # timm ViT (SigLIP)
    if hasattr(v, 'transformer') and hasattr(v.transformer, 'resblocks'):
        return list(v.transformer.resblocks)   # open_clip ViT
    return None

def set_trainable(net, n_blocks):
    for p in net.parameters():
        p.requires_grad = False
    blocks = visual_blocks(net)
    if blocks:
        for blk in blocks[-n_blocks:]:
            for p in blk.parameters():
                p.requires_grad = True
    else:
        print('WARN: could not find blocks; unfreezing whole visual tower')
        for p in net.visual.parameters():
            p.requires_grad = True
    # also the final norm / projection heads, if present
    for name in ('ln_post', 'norm', 'proj', 'head', 'attn_pool', 'trunk'):
        mod = getattr(net.visual, name, None)
        if name == 'trunk' and mod is not None:
            for nm in ('norm', 'head', 'attn_pool'):
                m2 = getattr(mod, nm, None)
                if isinstance(m2, torch.nn.Module):
                    for p in m2.parameters():
                        p.requires_grad = True
        elif isinstance(mod, torch.nn.Module):
            for p in mod.parameters():
                p.requires_grad = True
        elif isinstance(mod, torch.nn.Parameter):
            mod.requires_grad = True
    n = sum(p.numel() for p in net.parameters() if p.requires_grad)
    print(f'trainable params: {n:,}')

def supcon_loss(feats, w_batch, temp):
    sims = (feats @ feats.t()) / temp
    B = feats.size(0)
    eye = torch.eye(B, dtype=torch.bool, device=feats.device)
    sims = sims.masked_fill(eye, torch.finfo(feats.dtype).min)
    log_prob = sims - torch.logsumexp(sims, dim=1, keepdim=True)
    denom = w_batch.sum(1)
    valid = denom > 0
    if valid.sum() == 0:
        return None
    loss = -(w_batch * log_prob).sum(1)[valid] / denom[valid]
    return loss.mean()

def finetune(arm):
    model.load_state_dict(INIT_STATE)                      # reset to pretrained
    set_trainable(model, UNFREEZE_BLOCKS)
    row_des, Wdd, keys = build_designer_weights(records, arm)
    Wdd_t = torch.tensor(Wdd, dtype=torch.float32, device=device)
    des_t = torch.tensor(row_des, dtype=torch.long, device=device)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                            lr=LR, weight_decay=WEIGHT_DECAY)
    rng = np.random.default_rng(SEED)
    fit = np.asarray(fit_rows)

    best_val = base['top1']            # floor = base; never keep something worse on val
    best_state = copy.deepcopy(model.state_dict())
    stale = 0
    for epoch in range(EPOCHS):
        model.train()
        rng.shuffle(fit)
        tot = 0.0; nb = 0
        for i in range(0, len(fit), BATCH):
            rows = fit[i:i+BATCH]
            if len(rows) < 4:
                continue
            x = torch.stack([preprocess(pil_images[r]) for r in rows]).to(device)
            f = model.encode_image(x)
            f = f / f.norm(dim=-1, keepdim=True).clamp(min=1e-8)
            di = des_t[torch.tensor(rows, device=device)]
            Wb = Wdd_t[di][:, di].clone()
            Wb.fill_diagonal_(0.0)
            loss = supcon_loss(f, Wb, TEMP)
            if loss is None:
                continue
            opt.zero_grad(); loss.backward(); opt.step()
            tot += float(loss); nb += 1
        emb = embed_all(model)
        val = designer_topk(emb, designers, fit_rows, val_rows)['top1']
        print(f'[{arm}] epoch {epoch+1}/{EPOCHS} loss={tot/max(1,nb):.4f} val_top1={val:.3f}')
        if val > best_val + 1e-4:
            best_val = val
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
            if stale >= PATIENCE:
                print(f'[{arm}] early stop (best val_top1={best_val:.3f})')
                break
    model.load_state_dict(best_state)
    ft_emb = embed_all(model)
    return ft_emb, best_state, best_val

## 8. Run the ablation + the comparison table

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
results = {'base': {**{k: base[k] for k in ('top1','top3','top5','n_test')},
                    'ci95': bootstrap_ci(base['hit1'])}}
arms = ['concepts', 'labels'] if SUPERVISION == 'both' else [SUPERVISION]
for arm in arms:
    ft_emb, state, best_val = finetune(arm)
    res = designer_topk(ft_emb, designers, ref_rows, test_rows)
    results[arm] = {**{k: res[k] for k in ('top1','top3','top5','n_test')},
                    'ci95': bootstrap_ci(res['hit1'])}
    torch.save(state, f'{OUTPUT_DIR}/encoder_{arm}.pt')
    np.savez(f'{OUTPUT_DIR}/runway_{arm}_emb.npz', embeddings=ft_emb)
    print(f'saved encoder_{arm}.pt + embeddings')

print('\n=== Designer top-k (honest by-collection test split) ===')
print(f"{'condition':<10}{'top1':>7}{'  95% CI':>16}{'top3':>7}{'top5':>7}   n")
for k in [c for c in ('base','labels','concepts') if c in results]:
    r = results[k]; ci = f"[{r['ci95'][0]:.3f},{r['ci95'][1]:.3f}]"
    print(f"{k:<10}{r['top1']:>7.3f}{ci:>16}{r['top3']:>7.3f}{r['top5']:>7.3f}   {r['n_test']}")
if 'concepts' in results and 'labels' in results:
    print(f"\nKG-concept vs label-only \u0394top1 = {results['concepts']['top1']-results['labels']['top1']:+.3f}")
    print(f"KG-concept vs base       \u0394top1 = {results['concepts']['top1']-results['base']['top1']:+.3f}")

import json as _json
with open(f'{OUTPUT_DIR}/results.json', 'w') as fh:
    _json.dump({k: {kk: vv for kk, vv in v.items()} for k, v in results.items()}, fh, indent=2, default=list)
print('\nsaved', f'{OUTPUT_DIR}/results.json')

## 9. How to read it & what to do next

**Win condition (pre-registered):** `concepts` beats *both* `base` (0.411) *and* `labels` on
top-1, with CIs that don't fully overlap. That's the thesis result — KG structure, not just the
designer label, improved the *representation*.

**If it wins:** the fine-tuned encoder weights are in `phase5_out/encoder_concepts.pt` on your
Drive. Locally, load them into `FashionEmbedder` (`model.load_state_dict(torch.load(...))`),
rebuild the runway index (`fgraph vision build-runway`), and re-run `fgraph vision eval-runway`
to confirm — then the sharper embedder lifts RunwayLinker / RAG / Path A too.

**If it doesn't move (still inside base CI):** that's a strong, honest finding — even a fine-
tuned encoder on ~1.9k looks can't beat the pretrained fashion features for cross-collection
designer ID; report base 0.411 + the ablation. Levers to try first: raise `UNFREEZE_BLOCKS`
(more capacity), lower `LR`, or switch the metric to concept-retrieval (where concept alignment
is the fairer test of grounding).